# Early Sepsis Prediction Experiment — MIMIC-III Sepsis-3

Kaggle notebook for RNN, LSTM, CNN-Transformer and LSTM-Transformer; unweighted vs weighted BCE; 5-fold CV; held-out evaluation; Kernel SHAP; Temporal SHAP; Transformer attention; Streamlit export.

In [ ]:
MODE='pilot'; SEED=42; HORIZONS=[4,8]; MODELS=['rnn','lstm','cnn_transformer','lstm_transformer']; STRATEGIES=['unweighted','weighted']
PILOT_EPOCHS=3; FINAL_EPOCHS=20; BATCH_SIZE=128; LR=1e-3; WD=1e-4; HIDDEN=64; DROPOUT=.2; PATIENCE=5; CV_FOLDS=5
RUN_LOCAL_SHAP=False; RUN_TEMPORAL_SHAP=False; RUN_ATTENTION=False; XAI_HORIZON=4; XAI_MODEL='lstm_transformer'; XAI_STRATEGY='weighted'

In [ ]:
from pathlib import Path
from copy import deepcopy
import json,math,random,pickle,shutil,warnings
import numpy as np,pandas as pd,matplotlib.pyplot as plt
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split,StratifiedKFold
from sklearn.dummy import DummyClassifier
from sklearn.metrics import *
import torch
from torch import nn
from torch.utils.data import TensorDataset,DataLoader
warnings.filterwarnings('ignore'); random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu'); print(DEVICE)

def locate_data():
    c=[Path.cwd()/'data',Path('/kaggle/working/sepsis_test/data'),Path('/kaggle/input/sepsis-test/data')]
    if Path('/kaggle/input').exists():
        c += [q for p in Path('/kaggle/input').glob('*') for q in (p,p/'data')]
    for p in c:
        if (p/'dataset_manifest.json').exists(): return p
    raise FileNotFoundError('Attach repository data.')
DATA=locate_data(); P=DATA/'processed'; OUT=Path('/kaggle/working/sepsis_experiment'); OUT.mkdir(parents=True,exist_ok=True)
data={}
for h in HORIZONS:
    z=np.load(P/f'timeseries_before_onset_{h}h.npz',allow_pickle=False); data[h]={k:z[k] for k in z.files}; z.close()
print({h:(d['x'].shape,int(d['y'].sum())) for h,d in data.items()})

## Leakage-safe preprocessing and patient-disjoint split
Completely unobserved windows are removed. Remaining NaNs are imputed and standardized using training data only.

In [ ]:
clean={}; splits={}; proc={}; prep={}
for h,d in data.items():
 x=d['x'].astype('float32'); y=d['y'].astype(int); keep=~np.isnan(x).all((1,2)); dd={k:(v[keep] if isinstance(v,np.ndarray) and v.ndim and v.shape[0]==len(y) else v) for k,v in d.items()}; clean[h]=dd; x=dd['x'].astype('float32'); y=dd['y'].astype(int); sid=dd['subject_id']; idx=np.arange(len(y)); dev,test=train_test_split(idx,test_size=.30,random_state=SEED,stratify=y); tr,va=train_test_split(dev,test_size=.20,random_state=SEED,stratify=y[dev]); assert set(sid[tr]).isdisjoint(set(sid[va])|set(sid[test])); splits[h]={'train':tr,'val':va,'dev':dev,'test':test}; f=x.shape[2]; imp=SimpleImputer(strategy='mean',keep_empty_features=True); sc=StandardScaler(); sc.fit(imp.fit_transform(x[tr].reshape(-1,f))); tx=lambda a: sc.transform(imp.transform(a.reshape(-1,f))).reshape(a.shape).astype('float32'); prep[h]=(imp,sc); proc[h]={'x_train':tx(x[tr]),'y_train':y[tr],'x_val':tx(x[va]),'y_val':y[va],'x_test':tx(x[test]),'y_test':y[test]}
 print(h,[(n,len(i),int(y[i].sum())) for n,i in [('train',tr),('val',va),('test',test)]])

## Models and explainable Transformer attention

In [ ]:
class PE(nn.Module):
 def __init__(self,d):
  super().__init__(); pos=torch.arange(300).float()[:,None]; div=torch.exp(torch.arange(0,d,2).float()*(-math.log(10000)/d)); pe=torch.zeros(300,d); pe[:,0::2]=torch.sin(pos*div); pe[:,1::2]=torch.cos(pos*div[:pe[:,1::2].shape[1]]); self.register_buffer('pe',pe[None])
 def forward(self,x): return x+self.pe[:,:x.size(1)]
class IEL(nn.TransformerEncoderLayer):
 def __init__(self,*a,**k): super().__init__(*a,**k); self.capture=False; self.attn=None
 def _sa_block(self,x,attn_mask,key_padding_mask,is_causal=False):
  o,w=self.self_attn(x,x,x,attn_mask=attn_mask,key_padding_mask=key_padding_mask,need_weights=self.capture,average_attn_weights=False,is_causal=is_causal); self.attn=w.detach() if w is not None else None; return self.dropout1(o)
class Mix:
 def forward_attention(self,x):
  for l in self.encoder.layers: l.capture=True
  y=self.forward(x); maps=[l.attn for l in self.encoder.layers if l.attn is not None]
  for l in self.encoder.layers: l.capture=False
  return y,maps
class RNN(nn.Module):
 def __init__(self,f): super().__init__(); self.r=nn.RNN(f,HIDDEN,batch_first=True); self.h=nn.Linear(HIDDEN,1)
 def forward(self,x): return self.h(self.r(x)[1][-1]).squeeze(-1)
class LSTM(nn.Module):
 def __init__(self,f): super().__init__(); self.r=nn.LSTM(f,HIDDEN,batch_first=True); self.h=nn.Linear(HIDDEN,1)
 def forward(self,x): return self.h(self.r(x)[1][0][-1]).squeeze(-1)
class CT(Mix,nn.Module):
 def __init__(self,f): super().__init__(); self.c=nn.Conv1d(f,HIDDEN,3,padding=1); self.p=PE(HIDDEN); self.encoder=nn.TransformerEncoder(IEL(HIDDEN,4,HIDDEN*4,DROPOUT,batch_first=True),2); self.h=nn.Linear(HIDDEN,1)
 def forward(self,x): x=self.c(x.transpose(1,2)).transpose(1,2); return self.h(self.encoder(self.p(x)).mean(1)).squeeze(-1)
class LT(Mix,nn.Module):
 def __init__(self,f): super().__init__(); self.r=nn.LSTM(f,HIDDEN,batch_first=True); self.p=PE(HIDDEN); self.encoder=nn.TransformerEncoder(IEL(HIDDEN,4,HIDDEN*4,DROPOUT,batch_first=True),2); self.h=nn.Linear(HIDDEN,1)
 def forward(self,x): x,_=self.r(x); return self.h(self.encoder(self.p(x)).mean(1)).squeeze(-1)
def build(n,f): return {'rnn':RNN,'lstm':LSTM,'cnn_transformer':CT,'lstm_transformer':LT}[n](f)

In [ ]:
def predict(m,x):
 m.eval(); out=[]
 with torch.no_grad():
  for (xb,) in DataLoader(TensorDataset(torch.tensor(x).float()),batch_size=BATCH_SIZE): out.append(torch.sigmoid(m(xb.to(DEVICE))).cpu().numpy())
 return np.concatenate(out)
def pack(y,s):
 p=(s>=.5).astype(int); tn,fp,fn,tp=confusion_matrix(y,p,labels=[0,1]).ravel(); return {'accuracy':accuracy_score(y,p),'precision':precision_score(y,p,zero_division=0),'recall':recall_score(y,p,zero_division=0),'specificity':tn/(tn+fp) if tn+fp else np.nan,'f1':f1_score(y,p,zero_division=0),'balanced_accuracy':balanced_accuracy_score(y,p),'auroc':roc_auc_score(y,s),'aupr':average_precision_score(y,s),'brier':brier_score_loss(y,s)}
def train(name,xtr,ytr,xv,yv,epochs,strategy):
 m=build(name,xtr.shape[2]).to(DEVICE); pos=max(1,int(ytr.sum())); neg=max(1,int((ytr==0).sum())); loss=nn.BCEWithLogitsLoss(pos_weight=torch.tensor([neg/pos],device=DEVICE)) if strategy=='weighted' else nn.BCEWithLogitsLoss(); opt=torch.optim.AdamW(m.parameters(),lr=LR,weight_decay=WD); dl=DataLoader(TensorDataset(torch.tensor(xtr).float(),torch.tensor(ytr).float()),batch_size=BATCH_SIZE,shuffle=True); best=None; bv=1e99; pat=0
 for ep in range(epochs):
  m.train()
  for xb,yb in dl:
   xb,yb=xb.to(DEVICE),yb.to(DEVICE); opt.zero_grad(); z=loss(m(xb),yb); z.backward(); torch.nn.utils.clip_grad_norm_(m.parameters(),1); opt.step()
  m.eval(); vl=float(loss(m(torch.tensor(xv).float().to(DEVICE)),torch.tensor(yv).float().to(DEVICE)).detach().cpu())
  if vl<bv: bv=vl; best=deepcopy(m.state_dict()); pat=0
  else: pat+=1
  if pat>=PATIENCE: break
 m.load_state_dict(best); return m

## Main experiment
`final` mode trains all architectures with both imbalance strategies and performs 5-fold CV on the development partition.

In [ ]:
epochs=PILOT_EPOCHS if MODE=='pilot' else FINAL_EPOCHS; trained={}; rows=[]; preds=[]; cvrows=[]
for h,p in proc.items():
 trained[h]={s:{} for s in STRATEGIES}
 for strategy in STRATEGIES:
  for name in MODELS:
   m=train(name,p['x_train'],p['y_train'],p['x_val'],p['y_val'],epochs,strategy); trained[h][strategy][name]=m; score=predict(m,p['x_test']); r=pack(p['y_test'],score); r.update({'horizon':h,'model':name,'imbalance_strategy':strategy}); rows.append(r); ii=splits[h]['test']; d=clean[h]; preds.append(pd.DataFrame({'horizon':h,'model':name,'imbalance_strategy':strategy,'subject_id':d['subject_id'][ii],'icustay_id':d['icustay_id'][ii],'true_label':p['y_test'],'predicted_probability':score}))
   if MODE=='final':
    x=d['x'].astype('float32'); y=d['y'].astype(int); dev=splits[h]['dev']; sk=StratifiedKFold(CV_FOLDS,shuffle=True,random_state=SEED)
    for fold,(a,b) in enumerate(sk.split(x[dev],y[dev]),1):
     ti,vi=dev[a],dev[b]; f=x.shape[2]; imp=SimpleImputer(strategy='mean',keep_empty_features=True); sc=StandardScaler(); sc.fit(imp.fit_transform(x[ti].reshape(-1,f))); T=lambda q: sc.transform(imp.transform(q.reshape(-1,f))).reshape(q.shape).astype('float32'); mm=train(name,T(x[ti]),y[ti],T(x[vi]),y[vi],epochs,strategy); rr=pack(y[vi],predict(mm,T(x[vi]))); rr.update({'horizon':h,'model':name,'imbalance_strategy':strategy,'fold':fold}); cvrows.append(rr)
results_df=pd.DataFrame(rows); predictions_df=pd.concat(preds,ignore_index=True); results_df.to_csv(OUT/'model_metrics.csv',index=False); predictions_df.to_csv(OUT/'heldout_predictions.csv',index=False); display(results_df.round(4))
if cvrows: pd.DataFrame(cvrows).to_csv(OUT/'cross_validation_metrics.csv',index=False)

## Explainable AI — Local Kernel SHAP, Temporal SHAP and Transformer Attention
Enable the XAI flags after the final run. The same selected patient is used for all explanations and artifacts are exported for Streamlit.

In [ ]:
if RUN_LOCAL_SHAP or RUN_TEMPORAL_SHAP or RUN_ATTENTION:
 import shap
 h=XAI_HORIZON; m=trained[h][XAI_STRATEGY][XAI_MODEL]; p=proc[h]; d=clean[h]; ti=splits[h]['test']; scs=predict(m,p['x_test']); pp=np.where(p['y_test']==1)[0]; loc=int(pp[np.argmax(scs[pp])]) if len(pp) else int(np.argmax(scs)); oi=int(ti[loc]); raw=d['x'][oi].astype('float32'); names=d['feature_names'].astype(str).tolist(); imp,sca=prep[h]; bgidx=np.random.default_rng(SEED).choice(splits[h]['train'],min(20,len(splits[h]['train'])),replace=False); bg=d['x'][bgidx].astype('float32'); sid=int(d['subject_id'][oi]); pd.DataFrame([{'horizon':h,'model':XAI_MODEL,'imbalance_strategy':XAI_STRATEGY,'subject_id':sid,'icustay_id':int(d['icustay_id'][oi]),'true_label':int(p['y_test'][loc]),'predicted_probability':float(scs[loc])}]).to_csv(OUT/'xai_case.csv',index=False)
 def TX(a): f=a.shape[2]; return sca.transform(imp.transform(a.reshape(-1,f))).reshape(a.shape).astype('float32')
 if RUN_LOCAL_SHAP:
  ps=TX(raw[None]); bs=TX(bg); t,f=ps.shape[1:]; fn=lambda q: predict(m,np.asarray(q,dtype='float32').reshape(-1,t,f)); sv=np.asarray(shap.KernelExplainer(fn,bs.reshape(len(bs),-1)).shap_values(ps.reshape(1,-1),nsamples=100,silent=True)); sv=sv[0] if sv.ndim==3 else sv; sv=sv.reshape(t,f); tab=pd.DataFrame({'feature':names,'shap_contribution':sv.sum(0),'absolute_contribution':np.abs(sv.sum(0))}).sort_values('absolute_contribution',ascending=False); tab.to_csv(OUT/'local_shap_feature_contributions.csv',index=False); display(tab.head(15))
 if RUN_TEMPORAL_SHAP:
  tr=[]
  for k in range(1,13):
   ps=TX(raw[None,:k]); bs=TX(bg[:,:k]); _,t,f=ps.shape; fn=lambda q: predict(m,np.asarray(q,dtype='float32').reshape(-1,t,f)); v=np.asarray(shap.KernelExplainer(fn,bs.reshape(len(bs),-1)).shap_values(ps.reshape(1,-1),nsamples=50,silent=True)); v=v[0] if v.ndim==3 else v; v=v.reshape(t,f).sum(0); tr += [{'prefix_hour':k,'feature':names[j],'shap_contribution':float(v[j]),'absolute_contribution':float(abs(v[j]))} for j in range(f)]
  pd.DataFrame(tr).to_csv(OUT/'temporal_shap.csv',index=False)
 if RUN_ATTENTION:
  if XAI_MODEL not in ('cnn_transformer','lstm_transformer'): raise ValueError('Transformer model required')
  with torch.no_grad(): _,maps=m.forward_attention(torch.tensor(p['x_test'][loc:loc+1]).float().to(DEVICE))
  A=torch.stack([a[0].cpu() for a in maps]).numpy().mean((0,1)); pd.DataFrame(A).to_csv(OUT/'attention_matrix.csv'); pd.DataFrame({'hour':np.arange(1,A.shape[1]+1),'mean_attention_received':A.mean(0)}).to_csv(OUT/'attention_hour_importance.csv',index=False)
 print('XAI artifacts:',OUT)
else: print('XAI disabled until final model run.')

## Streamlit export
After the final run, download the ZIP and extract it into `results/` in GitHub. The deployed dashboard will then show model performance, patient predictions, Local SHAP, Temporal SHAP and attention.

In [ ]:
zip_path=shutil.make_archive(str(OUT/'streamlit_results'),'zip',root_dir=OUT); print(zip_path)